# PyTorch Tensor 常用用法：机器学习训练速查与练习

这份 Notebook 在原 Tensor 练习的基础上重新整理，保留并扩展了：

- Tensor 创建、属性、数据类型和设备；
- 索引、切片、条件筛选与高级索引；
- `reshape`、`view`、`flatten`、`squeeze`、`unsqueeze`、`permute`；
- 广播、逐元素运算、矩阵乘法与批量矩阵乘法；
- 聚合统计、拼接、拆分、排序与 Top-k；
- NumPy、Pandas 与 Tensor 的相互转换；
- CPU、GPU、`float16`、`bfloat16`；
- 自动求导、梯度累积、`detach()`、`no_grad()`；
- 图像、表格、序列数据在模型中的常见形状；
- 分类、回归、注意力计算中的 Tensor 操作；
- 使用纯 Tensor 完成一个最小训练循环；
- 保存、加载、随机种子、调试和性能注意事项。

> Tensor 是 PyTorch 中承载**数据、参数、中间特征、预测结果和梯度**的基本对象。  
> 学好 Tensor 的核心不是记住所有 API，而是始终看清四件事：  
> **shape、dtype、device、是否参与梯度计算。**

## 目录

1. Tensor 是什么  
2. 导入、随机种子与打印设置  
3. 创建 Tensor  
4. 查看属性  
5. 数据类型与类型转换  
6. 索引、切片和条件筛选  
7. 形状变换与内存连续性  
8. 逐元素运算与广播  
9. 矩阵乘法与批量矩阵运算  
10. 聚合统计、排序和 Top-k  
11. 拼接、堆叠和拆分  
12. 机器学习常见 Tensor 操作  
13. NumPy、Pandas 与 Tensor  
14. CPU、GPU 与低精度类型  
15. 自动求导与计算图  
16. 常见数据形状  
17. TensorDataset 与 DataLoader  
18. 最小训练循环  
19. 保存、加载与复现  
20. 调试、性能与常见错误  
21. 速查表与综合练习

## 1. Tensor 是什么

Tensor 可以理解为“支持 GPU 和自动求导的多维数组”。

| 维度 | 常见称呼 | 示例 shape |
|---|---|---|
| 0 维 | 标量 | `[]` |
| 1 维 | 向量 | `[features]` |
| 2 维 | 矩阵 | `[batch, features]` |
| 3 维 | 序列批次/彩色图片 | `[batch, length, hidden]` 或 `[channels, height, width]` |
| 4 维 | 图像批次 | `[batch, channels, height, width]` |

一个重要纠正：

> shape 为 `[n]` 的一维 Tensor **既不是行向量，也不是列向量**。  
> 要变成行向量使用 `[1, n]`，要变成列向量使用 `[n, 1]`。

## 2. 导入、随机种子与打印设置

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)

torch.set_printoptions(
    precision=4,
    sci_mode=False,
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.10.0+cpu
CUDA available: False


### 随机种子的作用

相同的随机种子可以让初始化和随机样本尽量一致，方便复现实验：

```python
torch.manual_seed(42)
```

还可以创建局部随机数生成器，避免修改全局随机状态。

In [2]:
generator = torch.Generator().manual_seed(123)

sample_1 = torch.rand(3, generator=generator)

generator = torch.Generator().manual_seed(123)
sample_2 = torch.rand(3, generator=generator)

print(sample_1)
print(sample_2)
print("是否相同:", torch.equal(sample_1, sample_2))

tensor([0.2961, 0.5166, 0.2517])
tensor([0.2961, 0.5166, 0.2517])
是否相同: True


## 3. 创建 Tensor

### 3.1 从 Python 数据创建

`torch.tensor()` 会根据数据推断类型，也可以显式指定 `dtype` 和 `device`。

In [3]:
integer_tensor = torch.tensor([1, 2, 3, 4])
float_tensor = torch.tensor(
    [1.0, 2.0, 3.0],
    dtype=torch.float32,
)
matrix = torch.tensor([
    [1.0, 2.0],
    [3.0, 4.0],
])

print(integer_tensor, integer_tensor.dtype)
print(float_tensor, float_tensor.dtype)
print(matrix, matrix.shape)

tensor([1, 2, 3, 4]) torch.int64
tensor([1., 2., 3.]) torch.float32
tensor([[1., 2.],
        [3., 4.]]) torch.Size([2, 2])


### 3.2 常见创建函数

In [4]:
zeros = torch.zeros(2, 3)
ones = torch.ones(2, 3)
full = torch.full((2, 3), fill_value=7.0)
identity = torch.eye(3)

random_uniform = torch.rand(2, 3)     # 均匀分布 U(0, 1)
random_normal = torch.randn(2, 3)     # 标准正态分布 N(0, 1)
random_integer = torch.randint(
    low=0,
    high=10,
    size=(2, 4),
)

print("zeros:\n", zeros)
print("ones:\n", ones)
print("full:\n", full)
print("identity:\n", identity)
print("uniform:\n", random_uniform)
print("normal:\n", random_normal)
print("integer:\n", random_integer)

zeros:
 tensor([[0., 0., 0.],
        [0., 0., 0.]])
ones:
 tensor([[1., 1., 1.],
        [1., 1., 1.]])
full:
 tensor([[7., 7., 7.],
        [7., 7., 7.]])
identity:
 tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])
uniform:
 tensor([[0.8823, 0.9150, 0.3829],
        [0.9593, 0.3904, 0.6009]])
normal:
 tensor([[ 1.1561,  0.3965, -2.4661],
        [ 0.3623,  0.3765, -0.1808]])
integer:
 tensor([[7, 6, 9, 6],
        [3, 1, 9, 3]])


### 3.3 创建等间隔序列

- `arange(start, end, step)`：类似 Python 的 `range`，右端点不包含；
- `linspace(start, end, steps)`：指定点的数量，通常包含两端点。

In [5]:
sequence = torch.arange(0, 12, 2)
linear_space = torch.linspace(0, 1, steps=6)

print("arange:", sequence)
print("linspace:", linear_space)

arange: tensor([ 0,  2,  4,  6,  8, 10])
linspace: tensor([0.0000, 0.2000, 0.4000, 0.6000, 0.8000, 1.0000])


### 3.4 `*_like()`：沿用已有 Tensor 的形状、类型和设备

模型代码中经常使用：

```python
torch.zeros_like(x)
torch.ones_like(x)
torch.rand_like(x)
torch.randn_like(x)
```

In [6]:
x = torch.tensor(
    [[1.0, 2.0], [3.0, 4.0]],
    dtype=torch.float64,
)

print("zeros_like:\n", torch.zeros_like(x))
print("rand_like dtype:", torch.rand_like(x).dtype)
print("rand_like device:", torch.rand_like(x).device)

zeros_like:
 tensor([[0., 0.],
        [0., 0.]], dtype=torch.float64)
rand_like dtype: torch.float64
rand_like device: cpu


### 3.5 随机排列

`torch.randperm(n)` 常用于打乱样本索引。

In [7]:
indices = torch.randperm(8)
print("random indices:", indices)

random indices: tensor([1, 6, 5, 0, 7, 3, 4, 2])


## 4. 查看 Tensor 属性

最值得优先检查的属性：

```python
x.shape
x.dtype
x.device
x.requires_grad
```

In [8]:
x = torch.randn(2, 3, 4)

print("shape:", x.shape)
print("size():", x.size())
print("dtype:", x.dtype)
print("device:", x.device)
print("ndim:", x.ndim)
print("numel:", x.numel())
print("requires_grad:", x.requires_grad)
print("is_contiguous:", x.is_contiguous())
print("stride:", x.stride())

shape: torch.Size([2, 3, 4])
size(): torch.Size([2, 3, 4])
dtype: torch.float32
device: cpu
ndim: 3
numel: 24
requires_grad: False
is_contiguous: True
stride: (12, 4, 1)


- `shape` / `size()`：每一维的长度；
- `ndim`：维度数量；
- `numel()`：元素总数；
- `stride()`：沿每一维移动一步时跨过多少个存储位置；
- `is_contiguous()`：内存布局是否连续。

## 5. 数据类型与类型转换

### 5.1 机器学习中常见的 dtype

| dtype | 常见用途 |
|---|---|
| `torch.float32` | 默认训练数据和模型参数 |
| `torch.float16` | GPU 混合精度训练 |
| `torch.bfloat16` | 支持设备上的低精度训练 |
| `torch.float64` | 高精度数值计算，深度学习中较少 |
| `torch.int64` / `torch.long` | 多分类标签、索引 |
| `torch.bool` | 条件掩码 |

In [9]:
x_int = torch.tensor([1, 2, 3])

x_float32 = x_int.float()
x_float16 = x_int.to(torch.float16)
x_float64 = x_int.double()
x_long = x_float32.long()
x_bool = x_int.bool()

print(x_float32.dtype)
print(x_float16.dtype)
print(x_float64.dtype)
print(x_long.dtype)
print(x_bool.dtype)

torch.float32
torch.float16
torch.float64
torch.int64
torch.bool


常用简写：

```python
x.float()    # float32
x.half()     # float16
x.double()   # float64
x.long()     # int64
x.int()      # int32
x.bool()     # bool
```

统一写法：

```python
x = x.to(dtype=torch.float32)
```

注意：浮点数转整数会直接截断小数部分，而不是四舍五入。

In [10]:
values = torch.tensor([1.2, 1.8, -1.8])

print("long():", values.long())
print("round():", values.round())

long(): tensor([ 1,  1, -1])
round(): tensor([ 1.,  2., -2.])


### 5.2 常见标签类型

多分类的 `CrossEntropyLoss` 通常要求：

- logits：浮点 Tensor，shape 为 `[batch, classes]`；
- 标签：`torch.long`，shape 为 `[batch]`。

In [11]:
logits = torch.randn(4, 3)
class_labels = torch.tensor([0, 2, 1, 0], dtype=torch.long)

loss = F.cross_entropy(logits, class_labels)

print("logits dtype:", logits.dtype)
print("labels dtype:", class_labels.dtype)
print("loss:", loss.item())

logits dtype: torch.float32
labels dtype: torch.int64
loss: 1.1750444173812866


## 6. 索引、切片和条件筛选

### 6.1 基础索引与切片

切片规则与 Python、NumPy 基本一致：

| 写法 | 含义 |
|---|---|
| `:` | 当前维度全部选择 |
| `a:b` | 从 `a` 到 `b-1` |
| `:b` | 从开头到 `b-1` |
| `a:` | 从 `a` 到末尾 |
| `a:b:step` | 按步长选取 |
| `-1` | 最后一个位置 |

In [12]:
x = torch.arange(12).reshape(3, 4)

print("x:\n", x)
print("第 0 行:", x[0])
print("第 1 列:", x[:, 1])
print("前两行、前两列:\n", x[:2, :2])
print("最后一个元素:", x[-1, -1].item())
print("隔一列取一次:\n", x[:, ::2])

x:
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
第 0 行: tensor([0, 1, 2, 3])
第 1 列: tensor([1, 5, 9])
前两行、前两列:
 tensor([[0, 1],
        [4, 5]])
最后一个元素: 11
隔一列取一次:
 tensor([[ 0,  2],
        [ 4,  6],
        [ 8, 10]])


索引一个位置通常会减少一个维度：

```python
x[0]       # shape 从 [3, 4] 变为 [4]
x[0:1]     # shape 保持为 [1, 4]
```

In [13]:
print("x[0].shape:", x[0].shape)
print("x[0:1].shape:", x[0:1].shape)

x[0].shape: torch.Size([4])
x[0:1].shape: torch.Size([1, 4])


### 6.2 布尔掩码

In [14]:
x = torch.tensor([1, 5, 2, 8, 3, 10])

mask = x > 4
selected = x[mask]

print("mask:", mask)
print("selected:", selected)

mask: tensor([False,  True, False,  True, False,  True])
selected: tensor([ 5,  8, 10])


条件可以组合：

```python
(x >= 3) & (x <= 8)
```

注意使用 `&`、`|`、`~`，并给每个条件加括号。

In [15]:
combined_mask = (x >= 3) & (x <= 8)
print(x[combined_mask])

tensor([5, 8, 3])


### 6.3 `torch.where()`：按条件选择

```python
torch.where(condition, value_if_true, value_if_false)
```

In [16]:
scores = torch.tensor([45.0, 72.0, 88.0, 59.0])
result = torch.where(
    scores >= 60,
    torch.tensor(1),
    torch.tensor(0),
)

print(result)

tensor([0, 1, 1, 0])


### 6.4 高级索引

使用索引 Tensor 选择指定位置：

In [17]:
x = torch.tensor([10, 20, 30, 40, 50])
chosen_indices = torch.tensor([4, 0, 2])

print(x[chosen_indices])
print(torch.index_select(x, dim=0, index=chosen_indices))

tensor([50, 10, 30])
tensor([50, 10, 30])


### 6.5 `gather()`：按每行不同位置取值

分类任务中经常用于取出每个样本真实类别对应的分数。

In [18]:
logits = torch.tensor([
    [1.2, 0.4, 2.0],
    [0.3, 1.8, 0.7],
])
labels = torch.tensor([2, 1])

selected_logits = logits.gather(
    dim=1,
    index=labels.unsqueeze(1),
)

print(selected_logits)
print("shape:", selected_logits.shape)

tensor([[2.0000],
        [1.8000]])
shape: torch.Size([2, 1])


## 7. 形状变换与内存连续性

### 7.1 `reshape()` 与 `view()`

两者都用于改变形状，元素总数必须保持不变。

- `view()` 要求底层内存布局兼容；
- `reshape()` 必要时会创建副本，通常更稳妥；
- `-1` 表示自动推断该维度。

In [19]:
x = torch.arange(12)

x_3_by_4 = x.reshape(3, 4)
x_2_by_6 = x.view(2, 6)
x_inferred = x.reshape(3, -1)

print("original:", x)
print("reshape(3, 4):\n", x_3_by_4)
print("view(2, 6):\n", x_2_by_6)
print("reshape(3, -1):\n", x_inferred)

original: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
reshape(3, 4):
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
view(2, 6):
 tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11]])
reshape(3, -1):
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])


### 7.2 `flatten()`

图像送入全连接层时，经常把除 batch 外的维度展平。

In [20]:
images = torch.randn(32, 1, 28, 28)

flattened_1 = images.reshape(images.shape[0], -1)
flattened_2 = images.flatten(start_dim=1)

print(flattened_1.shape)
print(flattened_2.shape)
print("结果相同:", torch.equal(flattened_1, flattened_2))

torch.Size([32, 784])
torch.Size([32, 784])
结果相同: True


### 7.3 `unsqueeze()` 与 `squeeze()`

- `unsqueeze(dim)`：插入长度为 1 的新维度；
- `squeeze(dim)`：删除指定的、长度为 1 的维度。

In [21]:
x = torch.tensor([1.0, 2.0, 3.0])

row_vector = x.unsqueeze(0)
column_vector = x.unsqueeze(1)

print("original:", x.shape)
print("row:", row_vector.shape)
print("column:", column_vector.shape)
print("squeezed:", column_vector.squeeze(1).shape)

original: torch.Size([3])
row: torch.Size([1, 3])
column: torch.Size([3, 1])
squeezed: torch.Size([3])


不要随意使用不带参数的 `squeeze()`：

```python
x.squeeze()
```

它会删除所有长度为 1 的维度。当 batch size 恰好为 1 时，可能意外删除 batch 维。

### 7.4 `transpose()`、`.T` 与 `permute()`

- 二维矩阵转置：`x.T` 或 `x.transpose(0, 1)`；
- 多维 Tensor 重排维度：`x.permute(...)`。

In [22]:
matrix = torch.arange(6).reshape(2, 3)
print("matrix:\n", matrix)
print("matrix.T:\n", matrix.T)

image_hwc = torch.randn(28, 28, 3)
image_chw = image_hwc.permute(2, 0, 1)

print("HWC:", image_hwc.shape)
print("CHW:", image_chw.shape)

matrix:
 tensor([[0, 1, 2],
        [3, 4, 5]])
matrix.T:
 tensor([[0, 3],
        [1, 4],
        [2, 5]])
HWC: torch.Size([28, 28, 3])
CHW: torch.Size([3, 28, 28])


### 7.5 连续性与 `contiguous()`

`transpose()` 和 `permute()` 通常只改变观察方式，不一定重新排列底层数据，因此结果可能不连续。

In [23]:
x = torch.arange(12).reshape(3, 4)
x_transposed = x.transpose(0, 1)

print("original contiguous:", x.is_contiguous())
print("transposed contiguous:", x_transposed.is_contiguous())

safe_view = x_transposed.contiguous().view(-1)
safe_reshape = x_transposed.reshape(-1)

print("contiguous().view:", safe_view)
print("reshape:", safe_reshape)

original contiguous: True
transposed contiguous: False
contiguous().view: tensor([ 0,  4,  8,  1,  5,  9,  2,  6, 10,  3,  7, 11])
reshape: tensor([ 0,  4,  8,  1,  5,  9,  2,  6, 10,  3,  7, 11])


### 7.6 `expand()` 与 `repeat()`

两者都能“扩大”数据：

- `expand()` 通常不复制底层数据，只创建广播视图；
- `repeat()` 真正重复数据，占用更多内存。

In [24]:
x = torch.tensor([[1.0], [2.0], [3.0]])

expanded = x.expand(3, 4)
repeated = x.repeat(1, 4)

print("expanded:\n", expanded)
print("repeated:\n", repeated)
print("expanded stride:", expanded.stride())
print("repeated stride:", repeated.stride())

expanded:
 tensor([[1., 1., 1., 1.],
        [2., 2., 2., 2.],
        [3., 3., 3., 3.]])
repeated:
 tensor([[1., 1., 1., 1.],
        [2., 2., 2., 2.],
        [3., 3., 3., 3.]])
expanded stride: (1, 0)
repeated stride: (4, 1)


## 8. 逐元素运算与广播

### 8.1 基础逐元素运算

In [25]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print("a + b:", a + b)
print("a - b:", a - b)
print("a * b:", a * b)
print("a / b:", a / b)
print("a ** 2:", a ** 2)
print("sqrt:", torch.sqrt(a))
print("exp:", torch.exp(a))
print("log:", torch.log(a))

a + b: tensor([5., 7., 9.])
a - b: tensor([-3., -3., -3.])
a * b: tensor([ 4., 10., 18.])
a / b: tensor([0.2500, 0.4000, 0.5000])
a ** 2: tensor([1., 4., 9.])
sqrt: tensor([1.0000, 1.4142, 1.7321])
exp: tensor([ 2.7183,  7.3891, 20.0855])
log: tensor([0.0000, 0.6931, 1.0986])


常见数学函数：

```python
torch.abs(x)
torch.sqrt(x)
torch.exp(x)
torch.log(x)
torch.sin(x)
torch.sigmoid(x)
torch.relu(x)
```

这些运算通常逐元素执行。

### 8.2 广播机制

广播会从最后一个维度开始比较：

- 两个维度相同，兼容；
- 其中一个维度为 1，兼容；
- 缺少的前置维度视为 1；
- 否则不能广播。

In [26]:
matrix = torch.tensor([
    [1, 2, 3],
    [4, 5, 6],
])
row = torch.tensor([10, 20, 30])
column = torch.tensor([[100], [200]])

print("matrix + row:\n", matrix + row)
print("matrix + column:\n", matrix + column)

matrix + row:
 tensor([[11, 22, 33],
        [14, 25, 36]])
matrix + column:
 tensor([[101, 102, 103],
        [204, 205, 206]])


广播是“逻辑扩展”，通常不需要先手动复制数据。

常见例子：

```python
features - mean
(features - mean) / std
logits + bias
attention_scores + mask
```

### 8.3 标准化示例

In [27]:
features = torch.tensor([
    [1.0, 10.0, 100.0],
    [2.0, 20.0, 120.0],
    [3.0, 30.0, 140.0],
    [4.0, 40.0, 160.0],
])

mean = features.mean(dim=0, keepdim=True)
std = features.std(dim=0, keepdim=True, unbiased=False)

standardized = (features - mean) / (std + 1e-8)

print("mean:", mean)
print("std:", std)
print("standardized:\n", standardized)
print("new mean:", standardized.mean(dim=0))

mean: tensor([[  2.5000,  25.0000, 130.0000]])
std: tensor([[ 1.1180, 11.1803, 22.3607]])
standardized:
 tensor([[-1.3416, -1.3416, -1.3416],
        [-0.4472, -0.4472, -0.4472],
        [ 0.4472,  0.4472,  0.4472],
        [ 1.3416,  1.3416,  1.3416]])
new mean: tensor([0.0000, 0.0000, 0.0000])


### 8.4 原地运算

以下划线结尾的方法通常是原地修改：

```python
x.add_(1)
x.zero_()
x.copy_(other)
```

原地操作节省部分内存，但可能破坏自动求导需要保存的中间结果。初学阶段优先使用非原地写法。

In [28]:
x = torch.tensor([1.0, 2.0, 3.0])
x.add_(10)

print(x)

tensor([11., 12., 13.])


## 9. 矩阵乘法与批量矩阵运算

### 9.1 逐元素乘法与矩阵乘法

```python
A * B       # 逐元素乘法
A @ B       # 矩阵乘法
torch.matmul(A, B)
```

In [29]:
A = torch.randn(3, 4)
B = torch.randn(4, 2)

C_1 = A @ B
C_2 = torch.matmul(A, B)

print("A shape:", A.shape)
print("B shape:", B.shape)
print("C shape:", C_1.shape)
print("结果近似相同:", torch.allclose(C_1, C_2))

A shape: torch.Size([3, 4])
B shape: torch.Size([4, 2])
C shape: torch.Size([3, 2])
结果近似相同: True


二维矩阵乘法的形状规则：

\[
[m, n] @ [n, p] 
ightarrow [m, p]
\]

中间维度必须相同。

### 9.2 向量点积

In [30]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print("dot:", torch.dot(a, b))
print("equivalent:", (a * b).sum())

dot: tensor(32.)
equivalent: tensor(32.)


### 9.3 批量矩阵乘法

`torch.bmm()` 只处理三维 Tensor：

\[
[b, m, n] @ [b, n, p] 
ightarrow [b, m, p]
\]

In [31]:
batch_A = torch.randn(5, 3, 4)
batch_B = torch.randn(5, 4, 2)

batch_C = torch.bmm(batch_A, batch_B)

print(batch_C.shape)

torch.Size([5, 3, 2])


`torch.matmul()` 支持更多维度，并可使用广播；`bmm()` 的 batch 维必须直接匹配。

### 9.4 注意力分数的形状示例

In [32]:
batch_size = 2
query_length = 4
key_length = 6
hidden_size = 8

queries = torch.randn(batch_size, query_length, hidden_size)
keys = torch.randn(batch_size, key_length, hidden_size)

attention_scores = queries @ keys.transpose(-2, -1)
attention_scores = attention_scores / hidden_size ** 0.5

print("queries:", queries.shape)
print("keys:", keys.shape)
print("attention scores:", attention_scores.shape)

queries: torch.Size([2, 4, 8])
keys: torch.Size([2, 6, 8])
attention scores: torch.Size([2, 4, 6])


等价的 `einsum` 写法：

```python
torch.einsum("bqd,bkd->bqk", queries, keys)
```

`einsum` 很灵活，但普通矩阵乘法能清楚表达时，优先使用 `@`。

In [33]:
attention_scores_einsum = torch.einsum(
    "bqd,bkd->bqk",
    queries,
    keys,
) / hidden_size ** 0.5

print(torch.allclose(
    attention_scores,
    attention_scores_einsum,
))

True


## 10. 聚合统计、排序和 Top-k

### 10.1 常见聚合

不指定 `dim` 时，对全部元素聚合；指定 `dim` 时，消去对应维度。

In [34]:
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
])

print("sum:", x.sum())
print("mean:", x.mean())
print("std:", x.std(unbiased=False))
print("min:", x.min())
print("max:", x.max())

print("每列均值:", x.mean(dim=0))
print("每行和:", x.sum(dim=1))

sum: tensor(21.)
mean: tensor(3.5000)
std: tensor(1.7078)
min: tensor(1.)
max: tensor(6.)
每列均值: tensor([2.5000, 3.5000, 4.5000])
每行和: tensor([ 6., 15.])


### 10.2 `dim` 的准确理解

`dim=k` 表示：

> 沿第 `k` 维进行聚合，并默认消去这一维。

对于 shape `[2, 3]`：

- `mean(dim=0)`：把两行压缩掉，得到每一列的均值，shape `[3]`；
- `mean(dim=1)`：把三列压缩掉，得到每一行的均值，shape `[2]`。

### 10.3 `keepdim=True`

保留长度为 1 的维度，方便后续广播。

In [35]:
row_mean = x.mean(dim=1, keepdim=True)

print(row_mean)
print("shape:", row_mean.shape)
print("centered:\n", x - row_mean)

tensor([[2.],
        [5.]])
shape: torch.Size([2, 1])
centered:
 tensor([[-1.,  0.,  1.],
        [-1.,  0.,  1.]])


### 10.4 最大值、索引与 `argmax()`

In [36]:
values, indices = x.max(dim=1)

print("每行最大值:", values)
print("每行最大值位置:", indices)
print("argmax:", x.argmax(dim=1))

每行最大值: tensor([3., 6.])
每行最大值位置: tensor([2, 2])
argmax: tensor([2, 2])


分类预测常写为：

```python
predictions = logits.argmax(dim=1)
```

### 10.5 排序与 Top-k

In [37]:
scores = torch.tensor([0.15, 0.72, 0.33, 0.91, 0.64])

sorted_values, sorted_indices = torch.sort(
    scores,
    descending=True,
)
top_values, top_indices = torch.topk(scores, k=3)

print("sorted values:", sorted_values)
print("sorted indices:", sorted_indices)
print("top values:", top_values)
print("top indices:", top_indices)

sorted values: tensor([0.9100, 0.7200, 0.6400, 0.3300, 0.1500])
sorted indices: tensor([3, 1, 4, 2, 0])
top values: tensor([0.9100, 0.7200, 0.6400])
top indices: tensor([3, 1, 4])


### 10.6 判断两个 Tensor 是否相同

- `torch.equal(a, b)`：shape 和每个值都完全相同；
- `torch.allclose(a, b)`：允许浮点误差。

In [38]:
a = torch.tensor([0.1 + 0.2])
b = torch.tensor([0.3])

print("equal:", torch.equal(a, b))
print("allclose:", torch.allclose(a, b))

equal: True
allclose: True


## 11. 拼接、堆叠和拆分

### 11.1 `cat()` 与 `stack()`

- `cat()`：沿已有维度拼接；
- `stack()`：创建新维度。

In [39]:
a = torch.ones(2, 3)
b = torch.zeros(2, 3)

concat_dim0 = torch.cat([a, b], dim=0)
concat_dim1 = torch.cat([a, b], dim=1)
stacked_dim0 = torch.stack([a, b], dim=0)

print("a shape:", a.shape)
print("cat dim=0:", concat_dim0.shape)
print("cat dim=1:", concat_dim1.shape)
print("stack dim=0:", stacked_dim0.shape)

a shape: torch.Size([2, 3])
cat dim=0: torch.Size([4, 3])
cat dim=1: torch.Size([2, 6])
stack dim=0: torch.Size([2, 2, 3])


`cat()` 要求除了拼接维度外，其他维度相同。  
`stack()` 要求所有 Tensor 的 shape 完全相同。

### 11.2 `split()`、`chunk()` 与 `unbind()`

In [40]:
x = torch.arange(24).reshape(6, 4)

split_parts = torch.split(x, split_size_or_sections=2, dim=0)
chunk_parts = torch.chunk(x, chunks=3, dim=0)
rows = torch.unbind(x, dim=0)

print("split shapes:", [part.shape for part in split_parts])
print("chunk shapes:", [part.shape for part in chunk_parts])
print("unbind number:", len(rows))
print("first unbound row:", rows[0])

split shapes: [torch.Size([2, 4]), torch.Size([2, 4]), torch.Size([2, 4])]
chunk shapes: [torch.Size([2, 4]), torch.Size([2, 4]), torch.Size([2, 4])]
unbind number: 6
first unbound row: tensor([0, 1, 2, 3])


- `split()`：按固定大小或指定大小拆分；
- `chunk()`：尽量拆成指定份数；
- `unbind()`：移除一个维度并返回多个 Tensor。

## 12. 机器学习常见 Tensor 操作

### 12.1 `clamp()`：限制数值范围

In [41]:
probabilities = torch.tensor([-0.2, 0.3, 0.9, 1.4])

print(torch.clamp(probabilities, min=0.0, max=1.0))

tensor([0.0000, 0.3000, 0.9000, 1.0000])


`clamp()` 常用于：

- 限制预测值范围；
- 防止某些手写计算出现极端值；
- 图像像素裁剪。

但训练损失应优先使用数值稳定的官方函数，而不是随意裁剪掩盖问题。

### 12.2 One-hot 编码

In [42]:
labels = torch.tensor([0, 2, 1, 2])
one_hot_labels = F.one_hot(
    labels,
    num_classes=3,
)

print(one_hot_labels)
print(one_hot_labels.dtype)

tensor([[1, 0, 0],
        [0, 0, 1],
        [0, 1, 0],
        [0, 0, 1]])
torch.int64


`F.one_hot()` 输出整数类型。需要参与浮点运算时：

```python
one_hot_labels.float()
```

`CrossEntropyLoss` 通常直接接收类别索引，不需要先做 One-hot。

### 12.3 Softmax、LogSoftmax 与预测类别

In [43]:
logits = torch.tensor([
    [2.0, 1.0, 0.1],
    [0.2, 1.4, 0.8],
])

probabilities = torch.softmax(logits, dim=1)
log_probabilities = torch.log_softmax(logits, dim=1)
predictions = logits.argmax(dim=1)

print("probabilities:\n", probabilities)
print("row sums:", probabilities.sum(dim=1))
print("log probabilities:\n", log_probabilities)
print("predictions:", predictions)

probabilities:
 tensor([[0.6590, 0.2424, 0.0986],
        [0.1628, 0.5405, 0.2967]])
row sums: tensor([1.0000, 1.0000])
log probabilities:
 tensor([[-0.4170, -1.4170, -2.3170],
        [-1.8152, -0.6152, -1.2152]])
predictions: tensor([0, 1])


- logits 是模型未归一化的输出；
- 多分类 Softmax 一般沿类别维 `dim=1`；
- 只为获得预测类别时，可以直接对 logits 使用 `argmax()`；
- `CrossEntropyLoss` 内部已经包含 LogSoftmax，不要提前再做 Softmax。

### 12.4 注意力掩码：`masked_fill()`

In [44]:
attention_scores = torch.tensor([
    [0.2, 0.7, 0.4, 0.9],
    [0.1, 0.3, 0.8, 0.5],
])
valid_mask = torch.tensor([
    [True, True, False, False],
    [True, True, True, False],
])

masked_scores = attention_scores.masked_fill(
    ~valid_mask,
    float("-inf"),
)
attention_weights = torch.softmax(masked_scores, dim=1)

print("masked scores:\n", masked_scores)
print("attention weights:\n", attention_weights)

masked scores:
 tensor([[0.2000, 0.7000,   -inf,   -inf],
        [0.1000, 0.3000, 0.8000,   -inf]])
attention weights:
 tensor([[0.3775, 0.6225, 0.0000, 0.0000],
        [0.2361, 0.2884, 0.4755, 0.0000]])


### 12.5 二分类与多分类的常见 shape

In [45]:
# 二分类：每个样本输出一个 logit
binary_logits = torch.randn(5)
binary_labels = torch.tensor(
    [1, 0, 1, 1, 0],
    dtype=torch.float32,
)
binary_loss = F.binary_cross_entropy_with_logits(
    binary_logits,
    binary_labels,
)

# 多分类：每个样本输出 C 个 logits
multiclass_logits = torch.randn(5, 3)
multiclass_labels = torch.tensor(
    [0, 2, 1, 0, 2],
    dtype=torch.long,
)
multiclass_loss = F.cross_entropy(
    multiclass_logits,
    multiclass_labels,
)

print("binary:", binary_logits.shape, binary_labels.shape)
print("multiclass:", multiclass_logits.shape, multiclass_labels.shape)
print("losses:", binary_loss.item(), multiclass_loss.item())

binary: torch.Size([5]) torch.Size([5])
multiclass: torch.Size([5, 3]) torch.Size([5])
losses: 0.9288085699081421 0.8738482594490051


## 13. NumPy、Pandas 与 Tensor

### 13.1 NumPy 转 Tensor

常见方法：

```python
torch.from_numpy(array)
torch.as_tensor(array)
torch.tensor(array)
```

区别：

- `from_numpy()`：通常共享 CPU 内存；
- `as_tensor()`：尽量避免复制；
- `tensor()`：通常复制一份新数据。

In [46]:
numpy_array = np.array(
    [1.0, 2.0, 3.0],
    dtype=np.float32,
)

shared_tensor = torch.from_numpy(numpy_array)
copied_tensor = torch.tensor(numpy_array)

numpy_array[0] = 100.0

print("NumPy:", numpy_array)
print("from_numpy:", shared_tensor)
print("torch.tensor:", copied_tensor)

NumPy: [100.   2.   3.]
from_numpy: tensor([100.,   2.,   3.])
torch.tensor: tensor([1., 2., 3.])


### 13.2 Tensor 转 NumPy

CPU Tensor：

```python
array = tensor.numpy()
```

更通用的模型代码：

```python
array = tensor.detach().cpu().numpy()
```

In [47]:
tensor = torch.tensor([1.0, 2.0, 3.0])
array = tensor.detach().cpu().numpy()

print(array, type(array))

[1. 2. 3.] <class 'numpy.ndarray'>


### 13.3 Pandas DataFrame 转 Tensor

DataFrame 通常先转 NumPy：

```python
df.to_numpy()
```

字符串列不能直接变成 Tensor，必须先删除或编码。

In [48]:
df = pd.DataFrame({
    "gender": [0, 1, 0],
    "score": [85, 92, 78],
    "study_hours": [5, 7, 3],
})

features = torch.from_numpy(
    df.to_numpy(dtype=np.float32)
)

print(features)
print(features.dtype)
print(features.shape)

tensor([[ 0., 85.,  5.],
        [ 1., 92.,  7.],
        [ 0., 78.,  3.]])
torch.float32
torch.Size([3, 3])


### 13.4 `clone()`、`detach()` 和复制关系

- `clone()`：复制数据，但保留梯度关系；
- `detach()`：切断计算图，通常仍共享底层存储；
- `detach().clone()`：切断计算图并复制数据。

In [49]:
x = torch.tensor([1.0, 2.0], requires_grad=True)
y = x * 3

cloned = y.clone()
detached = y.detach()
independent = y.detach().clone()

print("y requires_grad:", y.requires_grad)
print("clone requires_grad:", cloned.requires_grad)
print("detach requires_grad:", detached.requires_grad)
print("detach clone requires_grad:", independent.requires_grad)

y requires_grad: True
clone requires_grad: True
detach requires_grad: False
detach clone requires_grad: False


## 14. CPU、GPU 与低精度类型

### 14.1 选择设备

In [50]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

x = torch.randn(3, 4)
x = x.to(device)

print("selected device:", device)
print("x device:", x.device)

selected device: cpu
x device: cpu


模型参数和输入必须位于同一个设备：

```python
model = model.to(device)
inputs = inputs.to(device)
labels = labels.to(device)
```

不同设备上的 Tensor 不能直接参与运算。

### 14.2 同时指定 dtype 和 device

In [51]:
x = torch.tensor(
    [1.0, 2.0, 3.0],
    dtype=torch.float32,
    device=device,
)

y = x.to(
    dtype=torch.float64,
    device=device,
)

print(x.dtype, x.device)
print(y.dtype, y.device)

torch.float32 cpu
torch.float64 cpu


### 14.3 `float16` 与 `bfloat16`

```python
x = x.to(torch.float16)
x = x.half()
```

注意：

- `float16` 主要适合 GPU 上的混合精度训练；
- CPU 上部分运算可能不支持或更慢；
- `bfloat16` 精度较低，但指数范围接近 `float32`；
- 实际训练通常使用自动混合精度，而不是手动把所有数据和参数永久转换为 `float16`。

In [52]:
x_float32 = torch.tensor([1.0, 2.0, 3.0])
x_float16 = x_float32.to(torch.float16)
x_bfloat16 = x_float32.to(torch.bfloat16)

print(x_float32.dtype)
print(x_float16.dtype)
print(x_bfloat16.dtype)

torch.float32
torch.float16
torch.bfloat16


### 14.4 GPU Tensor 转 NumPy

GPU Tensor 不能直接调用 `.numpy()`：

```python
array = tensor.detach().cpu().numpy()
```

## 15. 自动求导与计算图

### 15.1 `requires_grad=True`

In [53]:
x = torch.tensor(
    2.0,
    requires_grad=True,
)

y = x ** 2 + 3 * x + 1
y.backward()

print("y:", y.item())
print("dy/dx:", x.grad.item())

y: 11.0
dy/dx: 7.0


因为：

\[
y=x^2+3x+1
\]

所以：

\[
\frac{dy}{dx}=2x+3
\]

当 \(x=2\) 时，梯度为 \(7\)。

### 15.2 向量输出为什么通常要先聚合

不带参数的 `backward()` 默认要求输出是标量。训练中的 Loss 通常就是标量。

In [54]:
x = torch.tensor(
    [1.0, 2.0, 3.0],
    requires_grad=True,
)

y = x ** 2 + 3 * x + 1
loss = y.sum()

loss.backward()

print("loss:", loss.item())
print("gradient:", x.grad)

loss: 35.0
gradient: tensor([5., 7., 9.])


也可以为向量输出显式传入外部梯度：

In [55]:
x = torch.tensor(
    [1.0, 2.0, 3.0],
    requires_grad=True,
)
y = x ** 2

y.backward(
    gradient=torch.tensor([1.0, 0.5, 0.1])
)

print(x.grad)

tensor([2.0000, 2.0000, 0.6000])


### 15.3 梯度会累积

每次调用 `backward()`，梯度默认累加到 `.grad`。

In [56]:
x = torch.tensor(2.0, requires_grad=True)

(x ** 2).backward()
print("first grad:", x.grad.item())

(x ** 2).backward()
print("accumulated grad:", x.grad.item())

x.grad.zero_()
print("after zero_:", x.grad.item())

first grad: 4.0
accumulated grad: 8.0
after zero_: 0.0


训练中常见：

```python
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

若忘记清零，梯度会跨 batch 累积。

### 15.4 叶子 Tensor 与中间 Tensor

In [57]:
x = torch.tensor(
    2.0,
    requires_grad=True,
)
y = x * 3
z = y ** 2

z.backward()

print("x is leaf:", x.is_leaf)
print("y is leaf:", y.is_leaf)
print("x.grad:", x.grad)
print("普通中间 Tensor 默认不保留 .grad")

x is leaf: True
y is leaf: False
x.grad: tensor(36.)
普通中间 Tensor 默认不保留 .grad


默认情况下，梯度主要保存在需要梯度的叶子 Tensor 中。  
确实需要查看中间 Tensor 梯度时，可在反向传播前调用：

```python
y.retain_grad()
```

In [58]:
x = torch.tensor(2.0, requires_grad=True)
y = x * 3
y.retain_grad()
z = y ** 2

z.backward()

print("x.grad:", x.grad)
print("y.grad:", y.grad)

x.grad: tensor(36.)
y.grad: tensor(12.)


### 15.5 `detach()`、`no_grad()` 与 `inference_mode()`

- `detach()`：得到不参与当前计算图的 Tensor；
- `torch.no_grad()`：临时关闭梯度记录；
- `torch.inference_mode()`：专门用于推理，通常开销更低。

In [59]:
x = torch.tensor(
    [1.0, 2.0, 3.0],
    requires_grad=True,
)

with torch.no_grad():
    y_no_grad = x * 2

with torch.inference_mode():
    y_inference = x * 2

y_detached = (x * 2).detach()

print(y_no_grad.requires_grad)
print(y_inference.requires_grad)
print(y_detached.requires_grad)

False
False
False


验证和测试阶段通常同时使用：

```python
model.eval()

with torch.inference_mode():
    predictions = model(inputs)
```

`model.eval()` 负责切换 Dropout、BatchNorm 等层的行为；  
`inference_mode()` 负责关闭梯度记录。两者作用不同。

### 15.6 `.item()` 的使用

`.item()` 只能把含有一个元素的 Tensor 转成 Python 数字。

In [60]:
loss = torch.tensor(0.1234)

print(loss.item())
print(type(loss.item()))

0.1234000027179718
<class 'float'>


不要在训练内层循环中过度调用 GPU Tensor 的 `.item()`，因为它可能触发 CPU 与 GPU 同步。通常先累计 Tensor，再按需要转换。

## 16. 常见数据形状

### 16.1 表格数据

```text
[batch_size, num_features]
```

In [61]:
tabular_features = torch.randn(32, 10)
tabular_labels = torch.randint(0, 2, (32,))

print(tabular_features.shape)
print(tabular_labels.shape)

torch.Size([32, 10])
torch.Size([32])


### 16.2 图像数据：NCHW

PyTorch 卷积层通常接收：

```text
[batch, channels, height, width]
```

In [62]:
batch_size = 32
images = torch.randn(batch_size, 1, 28, 28)
labels = torch.randint(
    low=0,
    high=10,
    size=(batch_size,),
)

flattened_images = images.flatten(start_dim=1)

print("images:", images.shape)
print("labels:", labels.shape)
print("flattened:", flattened_images.shape)

images: torch.Size([32, 1, 28, 28])
labels: torch.Size([32])
flattened: torch.Size([32, 784])


FashionMNIST：

- 单张灰度图：`[1, 28, 28]`；
- 一批 32 张：`[32, 1, 28, 28]`；
- 展平后：`[32, 784]`。

### 16.3 彩色图片通道顺序

许多图片库读取为 HWC：

```text
[height, width, channels]
```

PyTorch 卷积层通常需要 CHW：

```python
image_chw = image_hwc.permute(2, 0, 1)
```

### 16.4 序列和 Transformer 数据

常见 shape：

```text
[batch, sequence_length]
[batch, sequence_length, hidden_size]
```

In [63]:
token_ids = torch.randint(
    low=0,
    high=30000,
    size=(8, 128),
)
hidden_states = torch.randn(8, 128, 768)

print("token ids:", token_ids.shape)
print("hidden states:", hidden_states.shape)

token ids: torch.Size([8, 128])
hidden states: torch.Size([8, 128, 768])


### 16.5 回归标签 shape

模型输出可能是 `[batch, 1]`，标签最好也保持一致，避免意外广播。

In [64]:
predictions = torch.randn(16, 1)
targets = torch.randn(16, 1)

print("predictions:", predictions.shape)
print("targets:", targets.shape)
print("MSE:", F.mse_loss(predictions, targets).item())

predictions: torch.Size([16, 1])
targets: torch.Size([16, 1])
MSE: 2.101696014404297


若标签是 `[batch]` 而预测是 `[batch, 1]`，某些运算可能广播成错误的 `[batch, batch]`。训练前应主动检查 shape。

## 17. TensorDataset 与 DataLoader

当特征和标签已经是 Tensor 时，可直接使用 `TensorDataset`。

In [65]:
features = torch.randn(100, 5)
labels = torch.randint(0, 2, (100,))

dataset = TensorDataset(features, labels)
loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
)

batch_features, batch_labels = next(iter(loader))

print("dataset size:", len(dataset))
print("batch features:", batch_features.shape)
print("batch labels:", batch_labels.shape)

dataset size: 100
batch features: torch.Size([16, 5])
batch labels: torch.Size([16])


`TensorDataset` 要求各 Tensor 的第 0 维长度一致，因为第 0 维通常表示样本数量。

## 18. 使用纯 Tensor 完成最小训练循环

下面用自动求导拟合：

\[
y = 3x + 2 + \epsilon
\]

In [66]:
torch.manual_seed(42)

x_train = torch.linspace(-2, 2, steps=100).unsqueeze(1)
noise = torch.randn_like(x_train) * 0.2
y_train = 3 * x_train + 2 + noise

weight = torch.randn(
    1,
    requires_grad=True,
)
bias = torch.zeros(
    1,
    requires_grad=True,
)

learning_rate = 0.05
loss_history = []

for epoch in range(200):
    predictions = x_train * weight + bias
    loss = ((predictions - y_train) ** 2).mean()

    loss.backward()

    with torch.no_grad():
        weight -= learning_rate * weight.grad
        bias -= learning_rate * bias.grad

    weight.grad.zero_()
    bias.grad.zero_()

    loss_history.append(loss.item())

print("learned weight:", weight.item())
print("learned bias:", bias.item())
print("final loss:", loss_history[-1])

learned weight: 2.99705171585083
learned bias: 2.0119519233703613
final loss: 0.03851611539721489


这个训练循环包含训练模型最核心的 Tensor 操作：

1. 前向计算得到预测；
2. 计算标量 Loss；
3. `loss.backward()` 计算梯度；
4. 在 `no_grad()` 中更新参数；
5. 清空梯度；
6. 重复多个 Epoch。

实际项目中通常用 `nn.Module` 和 `torch.optim` 自动管理参数和更新。

## 19. 保存、加载与复现

### 19.1 保存和加载 Tensor

In [67]:
output_dir = Path("tensor_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

tensor_path = output_dir / "example_tensor.pt"
tensor_to_save = torch.arange(6).reshape(2, 3)

torch.save(tensor_to_save, tensor_path)
loaded_tensor = torch.load(
    tensor_path,
    weights_only=True,
)

print("loaded:\n", loaded_tensor)
print("equal:", torch.equal(tensor_to_save, loaded_tensor))

loaded:
 tensor([[0, 1, 2],
        [3, 4, 5]])
equal: True


可以保存字典：

```python
torch.save(
    {
        "features": features,
        "labels": labels,
        "epoch": epoch,
    },
    path,
)
```

模型项目中更常保存 `state_dict`，而不是直接保存整个模型对象。

### 19.2 随机种子与复现

常见设置：

```python
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
```

完全复现还受硬件、并行算法、库版本和数据加载方式影响，因此“设置种子”不等于在所有环境中绝对逐位一致。

## 20. 调试、性能与常见错误

### 20.1 训练前先检查四件事

```python
print(x.shape)
print(x.dtype)
print(x.device)
print(x.requires_grad)
```

对于标签，还要确认：

```python
print(labels.min(), labels.max())
```

In [68]:
features = torch.randn(8, 5)
labels = torch.tensor([0, 1, 1, 0, 1, 0, 1, 0])

assert features.ndim == 2
assert labels.ndim == 1
assert features.shape[0] == labels.shape[0]
assert labels.dtype == torch.long

print("basic checks passed")

basic checks passed


### 20.2 检查 NaN 和 Inf

In [69]:
values = torch.tensor([
    1.0,
    float("nan"),
    float("inf"),
])

print("isnan:", torch.isnan(values))
print("isinf:", torch.isinf(values))
print("isfinite:", torch.isfinite(values))
print("all finite:", torch.isfinite(values).all().item())

isnan: tensor([False,  True, False])
isinf: tensor([False, False,  True])
isfinite: tensor([ True, False, False])
all finite: False


训练中出现 NaN 时，常见原因包括：

- 学习率过大；
- 除以接近 0 的数；
- 对非正数取对数；
- `exp()` 输入过大；
- 梯度爆炸；
- 输入数据中已经含有 NaN 或 Inf。

### 20.3 向量化优先于 Python 循环

In [70]:
x = torch.arange(1, 6, dtype=torch.float32)

vectorized = x ** 2
loop_result = torch.tensor([
    value.item() ** 2
    for value in x
])

print(vectorized)
print(loop_result)
print(torch.equal(vectorized, loop_result))

tensor([ 1.,  4.,  9., 16., 25.])
tensor([ 1.,  4.,  9., 16., 25.])
True


在大量数据上，应优先使用 Tensor 运算，因为 PyTorch 才能充分利用底层并行计算和 GPU。

### 20.4 视图、复制和原地修改

某些切片、转置、`detach()` 和 `expand()` 可能与原 Tensor 共享底层存储。  
修改一个对象可能影响另一个对象。

需要独立副本时：

```python
independent = x.clone()
```

需要切断计算图并复制：

```python
independent = x.detach().clone()
```

In [71]:
original = torch.tensor([1.0, 2.0, 3.0])
view = original[:2]
copy = original[:2].clone()

view[0] = 100.0
copy[1] = 200.0

print("original:", original)
print("view:", view)
print("copy:", copy)

original: tensor([100.,   2.,   3.])
view: tensor([100.,   2.])
copy: tensor([  1., 200.])


### 20.5 常见报错快速定位

| 报错/现象 | 常见原因 | 检查方式 |
|---|---|---|
| shape mismatch | 矩阵维度或标签 shape 错误 | 打印 `.shape` |
| dtype mismatch | 标签不是 `long`，或整数参与浮点层 | 打印 `.dtype` |
| device mismatch | 模型与数据不在同一设备 | 打印 `.device` |
| `view` 报连续性错误 | 转置后 Tensor 不连续 | 用 `reshape()` 或 `contiguous()` |
| backward 第二次报错 | 计算图已释放 | 重新前向，必要时谨慎用 `retain_graph=True` |
| 梯度越来越大 | 忘记清零梯度 | `optimizer.zero_grad()` |
| NumPy 转换报错 | Tensor 在 GPU 或参与梯度 | `detach().cpu().numpy()` |
| Loss shape 警告 | 预测和标签 shape 不一致 | 同时打印两者 shape |
| 出现 NaN | 数值不稳定或输入异常 | `torch.isfinite()` |

### 20.6 数值稳定性

不要手写不稳定版本：

```python
prob = torch.softmax(logits, dim=1)
loss = -torch.log(prob[...])
```

优先使用：

```python
F.cross_entropy(logits, labels)
F.binary_cross_entropy_with_logits(logits, labels)
torch.logsumexp(x, dim=...)
```

这些函数通常做了更稳定的数学处理。

In [72]:
large_values = torch.tensor([1000.0, 1001.0, 1002.0])

stable_value = torch.logsumexp(
    large_values,
    dim=0,
)

print(stable_value)

tensor(1002.4076)


### 20.7 测试 Tensor 结果

In [73]:
actual = torch.tensor([0.30000001, 0.5])
expected = torch.tensor([0.3, 0.5])

torch.testing.assert_close(
    actual,
    expected,
)

print("assert_close passed")

assert_close passed


## 21. 快速速查表

### 21.1 创建与属性

| 目的 | 写法 |
|---|---|
| 从列表创建 | `torch.tensor(data)` |
| 全 0 / 全 1 | `torch.zeros(shape)` / `torch.ones(shape)` |
| 均匀随机 | `torch.rand(shape)` |
| 正态随机 | `torch.randn(shape)` |
| 随机整数 | `torch.randint(low, high, shape)` |
| 等差序列 | `torch.arange(start, end, step)` |
| 固定数量点 | `torch.linspace(start, end, steps)` |
| 沿用 x 配置 | `torch.zeros_like(x)` |
| 查看形状 | `x.shape` |
| 查看类型 | `x.dtype` |
| 查看设备 | `x.device` |
| 元素总数 | `x.numel()` |

### 21.2 形状和索引

| 目的 | 写法 |
|---|---|
| 改变形状 | `x.reshape(...)` |
| 兼容内存的视图 | `x.view(...)` |
| 展平 | `x.flatten(start_dim=1)` |
| 增加维度 | `x.unsqueeze(dim)` |
| 删除长度 1 的维度 | `x.squeeze(dim)` |
| 交换两个维度 | `x.transpose(dim0, dim1)` |
| 任意重排维度 | `x.permute(...)` |
| 条件筛选 | `x[x > 0]` |
| 条件选择 | `torch.where(mask, a, b)` |

### 21.3 运算与聚合

| 目的 | 写法 |
|---|---|
| 逐元素乘法 | `a * b` |
| 矩阵乘法 | `a @ b` |
| 批量矩阵乘法 | `torch.bmm(a, b)` |
| 求和 / 均值 | `x.sum(dim=...)` / `x.mean(dim=...)` |
| 最大值和位置 | `x.max(dim=...)` |
| 最大值位置 | `x.argmax(dim=...)` |
| 前 k 个 | `torch.topk(x, k)` |
| 限制范围 | `x.clamp(min, max)` |
| 拼接 | `torch.cat([...], dim=...)` |
| 新维度堆叠 | `torch.stack([...], dim=...)` |

### 21.4 转换、设备和梯度

| 目的 | 写法 |
|---|---|
| 转 float32 | `x.float()` |
| 转 float16 | `x.half()` |
| 转 int64 | `x.long()` |
| 改类型/设备 | `x.to(dtype=..., device=...)` |
| NumPy → Tensor | `torch.from_numpy(array)` |
| Tensor → NumPy | `x.detach().cpu().numpy()` |
| 独立复制 | `x.clone()` |
| 切断计算图 | `x.detach()` |
| 反向传播 | `loss.backward()` |
| 读取梯度 | `x.grad` |
| 关闭梯度 | `with torch.no_grad():` |
| 推理模式 | `with torch.inference_mode():` |

## 22. 综合练习

### 练习 1：表格数据

创建 shape 为 `[100, 8]` 的特征 Tensor 和 `[100]` 的二分类标签：

- 特征使用 `float32`；
- 标签使用 `long`；
- 检查样本数量是否一致；
- 使用 `TensorDataset` 和 `DataLoader` 得到一个 batch。

### 练习 2：图像形状

创建 `[16, 3, 32, 32]` 的彩色图片：

- 展平为 `[16, 3072]`；
- 计算每个通道的均值；
- 保留维度，使结果可以继续广播到原图片。

### 练习 3：分类输出

创建 `[8, 5]` 的 logits：

- 计算 Softmax 概率；
- 预测每个样本的类别；
- 取每个样本概率最高的两个类别；
- 验证每行概率之和约等于 1。

### 练习 4：广播

创建 `[4, 3]` 的特征矩阵和 `[3]` 的均值：

- 用广播完成中心化；
- 解释为什么不需要手动把均值复制 4 次。

### 练习 5：自动求导

令：

\[
y = 2x^3 - 5x + 1
\]

在 \(x=2\) 时：

- 使用 PyTorch 自动计算梯度；
- 手动求导进行核对；
- 调用两次 `backward()`，观察梯度累积；
- 使用 `zero_()` 清空梯度。

### 练习 6：调试

故意构造并修复：

- 预测 `[32, 1]`、标签 `[32]` 的 shape 问题；
- CPU Tensor 与其他设备 Tensor 的 device 问题；
- `float` 标签传给 `cross_entropy` 的 dtype 问题；
- 转置后直接调用 `view()` 的连续性问题。

## 最后总结

机器学习训练中，Tensor 最需要形成的习惯是：

1. 每个关键节点都明确 `shape`；
2. 特征通常是浮点类型，多分类标签通常是 `long`；
3. 模型、输入、标签必须位于同一设备；
4. `*` 是逐元素乘法，`@` 是矩阵乘法；
5. `dim` 表示沿哪一维聚合；
6. `reshape()`、`unsqueeze()`、`permute()` 是最常见的形状工具；
7. 训练 Loss 应为标量，再调用 `backward()`；
8. 梯度默认累积，训练时必须清零；
9. GPU Tensor 转 NumPy 使用 `detach().cpu().numpy()`；
10. 出错时优先检查 `shape、dtype、device、requires_grad`。

只要这四项始终清楚，大多数 Tensor 问题都能迅速定位。